# EDA – Task 1.14 (Trâm)

Notebook khung EDA trên **data freeze v1.0** (`data/processed/`, task 1.7 + 1.19). Chỉ đọc thư mục này, không đọc `crawl_data/`.

Đổi `FLOWS_SET = "A"` thành `"B"` ở ô đầu tiên nếu nhóm chốt phương án B, rồi chạy lại.

**Nội dung**
1. Tổng quan bảng dữ liệu
2. Chuỗi giá BTC (thang log, đánh dấu halving và ngày thiếu nến)
3. Chỉ số On-chain theo 4 nhóm
4. Bản đồ dữ liệu thiếu
5. Tương quan giữa các đặc trưng (OHLCV và On-chain)
6. Tỷ lệ nhãn tăng/giảm

> ⚠️ **Không vẽ tương quan giữa đặc trưng và nhãn y** (theo bảng phân công 1.14). Xem nhãn trước khi chốt
> đặc trưng là data snooping. Mục 6 chỉ đếm tỷ lệ nhãn, không đặt nhãn cạnh đặc trưng.

Chạy từ thư mục `tram_features_backtest/notebooks/`. Cần chạy `src/features_ohlcv.py` trước để có `data/features_ohlcv.csv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

ROOT = Path.cwd().resolve().parents[1]          # thư mục gốc repo
PROCESSED = ROOT / "data/processed"                # data freeze: nguồn dữ liệu duy nhất
FLOWS_SET = "A"                                     # "A": Netflow + Sply; "B": In + Out + Sply (chờ chốt 1.19)
FIG = ROOT / "tram_features_backtest/reports/figures"
FIG.mkdir(parents=True, exist_ok=True)

START, END = "2018-01-01", "2026-08-31"          # phạm vi nghiên cứu trong đề xuất
HALVINGS = ["2020-05-11", "2024-04-20"]

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

ModuleNotFoundError: No module named 'pandas'

## 1. Tổng quan bảng dữ liệu

In [ ]:
import json, hashlib
# Windows: Git đổi xuống dòng CSV sang CRLF khi checkout -> chuẩn hoá về LF trước khi so sha256 với MANIFEST
manifest = json.loads((PROCESSED / "MANIFEST.json").read_text(encoding="utf-8"))
for name in ["master_daily.csv", f"data_dictionary_{FLOWS_SET}.csv"]:
    assert hashlib.sha256((PROCESSED / name).read_bytes().replace(b"\r\n", b"\n")).hexdigest() == manifest["files"][name]["sha256"],         f"{name} không khớp MANIFEST: kéo bản mới nhất (git pull)"
print("Data freeze", manifest["version"], "| flows set", FLOWS_SET)

master = pd.read_csv(PROCESSED / "master_daily.csv", parse_dates=["date"]).set_index("date")
dic = pd.read_csv(PROCESSED / f"data_dictionary_{FLOWS_SET}.csv")
feat_ohlcv = pd.read_csv(ROOT / "tram_features_backtest/data/features_ohlcv.csv",
                         parse_dates=["date"]).set_index("date")

df = master.loc[START:END]
print("master_daily:", master.shape, "| trong phạm vi nghiên cứu:", df.shape)
print("Ngày liên tục, không trùng:", df.index.is_unique and (df.index.to_series().diff().dropna() == pd.Timedelta("1D")).all())
df.describe().T

In [ ]:
# Nhóm của từng cột On-chain lấy thẳng từ data dictionary của Khoa
# Cột đưa vào mô hình: status = retained, role là candidate hoặc backup (README của data freeze)
retained = dic[(dic["status"] == "retained") & dic["role"].isin(["candidate", "backup"])]
GROUPS = retained.groupby("group")["column"].apply(list).to_dict()
ONCHAIN = [c for g in ["activity", "flows", "mining", "valuation"] for c in GROUPS.get(g, [])]
SIGNED = [c for c in ONCHAIN if (master[c] <= 0).any()]   # cột có dấu (Netflow): không lấy log được
print(len(ONCHAIN), "chỉ số On-chain; cột có dấu:", SIGNED)
GROUPS

## 2. Chuỗi giá BTC

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df["ohlcv_close"], lw=0.9, color="tab:orange", label="Binance close")
ax.set_yscale("log")
for h in HALVINGS:
    ax.axvline(pd.Timestamp(h), color="grey", ls="--", lw=1)
    ax.text(pd.Timestamp(h), ax.get_ylim()[1], " halving", va="top", fontsize=8, color="grey")
inc = df[df["ohlcv_n_bars_1h"] < 24]
ax.scatter(inc.index, inc["ohlcv_close"], s=12, color="red", zorder=3,
           label=f"Ngày < 24 nến 1h ({len(inc)} ngày)")
ax.set_title("Giá đóng cửa BTC/USDT (thang log), 2018 – 08/2026")
ax.set_ylabel("USDT"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "eda_01_price.png")
plt.show()

In [ ]:
# Hai nguồn giá có khớp nhau không? (Binance vs giá tham chiếu CoinMetrics)
gap = (df["ohlcv_close"] / df["cm_PriceUSD"] - 1) * 100
print(gap.describe().round(3))

## 3. Chỉ số On-chain theo nhóm

Mỗi nhóm một hàng. Chuỗi mức (số địa chỉ, hash rate, …) vẽ thang log; chuỗi tỷ lệ (MVRV, Puell, NVT) vẽ thang thường.

In [ ]:
RATIO = {"cm_CapMVRVCur", "puell_multiple", "nvt_ratio_bc"}
order = ["activity", "flows", "mining", "valuation"]
ncol = max(len(GROUPS[g]) for g in order)
fig, axes = plt.subplots(len(order), ncol, figsize=(4.2 * ncol, 2.6 * len(order)), squeeze=False)
for i, g in enumerate(order):
    for j in range(ncol):
        ax = axes[i, j]
        if j >= len(GROUPS[g]):
            ax.axis("off"); continue
        c = GROUPS[g][j]
        ax.plot(df.index, df[c], lw=0.7)
        if c in SIGNED:
            ax.axhline(0, color="k", lw=0.6)
        elif c not in RATIO:
            ax.set_yscale("log")
        for h in HALVINGS:
            ax.axvline(pd.Timestamp(h), color="grey", ls="--", lw=0.8)
        ax.set_title(f"[{g}] {c}", fontsize=9)
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.tight_layout(); fig.savefig(FIG / "eda_02_onchain_groups.png")
plt.show()

## 4. Bản đồ dữ liệu thiếu

In [ ]:
cols = [c for c in master.columns if c != "ohlcv_n_bars_1h"]
miss = master[cols].isna()
fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(miss.T.values, aspect="auto", interpolation="nearest", cmap="Greys",
          extent=[mdates.date2num(master.index[0]), mdates.date2num(master.index[-1]), len(cols) - 0.5, -0.5])
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols, fontsize=8)
ax.xaxis_date(); ax.axvline(pd.Timestamp(START), color="red", lw=1)
ax.set_title("Ô đen = thiếu dữ liệu (toàn bộ 2016 – 2026; vạch đỏ = bắt đầu phạm vi nghiên cứu)")
ax.grid(False)
fig.tight_layout(); fig.savefig(FIG / "eda_03_missing_map.png")
plt.show()

summary = pd.DataFrame({"thiếu_toàn_bảng": master[cols].isna().sum(),
                        "thiếu_2018_2026": df[cols].isna().sum(),
                        "tỷ_lệ_2018_2026_%": (df[cols].isna().mean() * 100).round(2)})
summary[summary["thiếu_toàn_bảng"] > 0]

In [ ]:
# Các ngày cụ thể còn thiếu trong phạm vi nghiên cứu
{c: df.index[df[c].isna()].strftime("%Y-%m-%d").tolist() for c in cols if df[c].isna().any()}

## 5. Tương quan giữa các đặc trưng

Tính trên **log-diff** của On-chain (giống quy tắc QA của Khoa); cột có dấu như Netflow dùng **sai phân thường** x_t − x_{t−1}, giống `features.py` của Ngọc, và trên đặc trưng OHLCV từ `features_ohlcv.csv`.
Dùng Spearman để ít nhạy với ngoại lai. Chỉ để mô tả, **không** dùng để chọn đặc trưng.

In [ ]:
pos = [c for c in ONCHAIN if c not in SIGNED]
onchain_ld = pd.concat([np.log(df[pos]).diff(), df[SIGNED].diff()], axis=1)[ONCHAIN]
X = feat_ohlcv.loc[START:END].join(onchain_ld, how="inner")
corr = X.corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
n_ohlcv = feat_ohlcv.shape[1]
ax.axhline(n_ohlcv - 0.5, color="k", lw=1); ax.axvline(n_ohlcv - 0.5, color="k", lw=1)
fig.colorbar(im, ax=ax, shrink=0.8, label="Spearman ρ")
ax.set_title("Tương quan giữa các đặc trưng (OHLCV | On-chain log-diff), 2018 – 08/2026")
ax.grid(False)
fig.tight_layout(); fig.savefig(FIG / "eda_04_feature_corr.png")
plt.show()

In [ ]:
# Các cặp tương quan mạnh |ρ| > 0.7 (gợi ý đa cộng tuyến cần lưu ý khi đọc SHAP)
iu = np.triu_indices_from(corr, k=1)
pairs = pd.DataFrame({"a": corr.index[iu[0]], "b": corr.columns[iu[1]], "rho": corr.values[iu]})
pairs.loc[pairs["rho"].abs() > 0.7].sort_values("rho", key=abs, ascending=False)

## 6. Tỷ lệ nhãn tăng/giảm

y_{t+1} = 1 nếu close_{t+1} > close_t. Chỉ đếm tỷ lệ để biết hai lớp có lệch không
(ảnh hưởng tới baseline majority class và việc dùng MCC/F1 macro).

In [ ]:
close = master["ohlcv_close"].dropna()
y = (close.shift(-1) > close).astype(float).where(close.shift(-1).notna()).dropna()
y = y.loc[START:END]
print(f"Tỷ lệ ngày tăng toàn kỳ: {y.mean():.3f}  (n = {len(y)})")
by_year = y.groupby(y.index.year).agg(["mean", "size"]).rename(columns={"mean": "tỷ_lệ_tăng", "size": "số_ngày"})

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(by_year.index.astype(str), by_year["tỷ_lệ_tăng"], color="tab:blue")
ax.axhline(0.5, color="red", ls="--", lw=1)
ax.set_ylim(0.4, 0.6); ax.set_title("Tỷ lệ ngày giá tăng theo năm")
fig.tight_layout(); fig.savefig(FIG / "eda_05_label_balance.png")
plt.show()
by_year.round(3)

## Ghi chú cho nhóm

- **Độ trễ công bố On-chain:** README của Khoa ghi CoinMetrics công bố ngày *d* khoảng 2–9,5 giờ sau 00:00 UTC ngày *d+1*.
  Khi dự báo ở 00:00 UTC ngày t+1, chỉ được dùng On-chain đến ngày t−1 (lag ≥ 1). Cần thống nhất với Ngọc (task 1.16).
- **Ngày thiếu nến:** 2018-02-08 chỉ có 1 nến 1h nên `dlog_volume` ngày đó và ngày sau rất lớn. Nhóm cần quyết định có giữ những ngày `ohlcv_n_bars_1h < 24` trong train/test không.
- **Flows:** In, Out và Netflow phụ thuộc tuyến tính, không dùng cả ba cùng lúc (xem README của Vi).